# =========================================================
# ⚙️ Phase 3: Feature Engineering & Model Preparation
# =========================================================

OBJECTIVE:
This notebook transitions from exploratory analysis to predictive modeling by 
transforming raw indicators into model-ready features.

FOCUS AREAS:
- Target Engineering: Defining Mortality Risk Levels (Low, Medium, High).
- Temporal Signals: Creating Lag features to capture delayed economic impacts.
- Efficiency Metrics: Modeling the relationship between health spend and outcomes.

GOAL:
Prepare a high-quality, structured dataset for Machine Learning (Random Forest) 
to classify and predict global mortality risk.

### Step 1: Imports & Dataset Loading

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# LOAD DATASET
# Using the processed dataset from the data directory
file_path = 'data/processed/final_dataset.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Success: Dataset loaded from {file_path}")
    print(f"Dataset Shape: {df.shape}")
except FileNotFoundError:
    print("Error: File not found. Please verify the path: 'data/processed/final_dataset.csv'")

# PREVIEW DATA
# Checking first 5 rows to ensure columns are intact
df.head()

Success: Dataset loaded from data/processed/final_dataset.csv
Dataset Shape: (4752, 8)


,Country Name,Year,Mortality_Rate,GDP_per_capita,Health_Expenditure,Homicide_Rate,Food_Price_Index,Food_Price_Log
0,Angola,2000,18.080,563.733796,1.685253,0.00000,140.0,4.94876
1,Albania,2000,5.619,1160.420471,5.874297,4.13752,140.0,4.94876
2,Andorra,2000,6.228,21810.250381,5.933423,0.00000,140.0,4.94876
3,United Arab Emirates,2000,2.004,29865.502347,2.395045,0.00000,140.0,4.94876
4,Argentina,2000,7.570,7637.014892,8.220011,0.00000,140.0,4.94876


### Step 2: Target Variable Engineering (Risk Classification)

In [4]:
# CONSTRUCT TARGET VARIABLE
# Discretizing Mortality Rate into 3 categories: Low, Medium, and High Risk
# We use qcut to ensure equal distribution across classes for better model training
df['Mortality_Risk'] = pd.qcut(df['Mortality_Rate'], q=3, labels=['Low', 'Medium', 'High'])

# VALIDATE CLASS BALANCE
# Checking how many countries fall into each risk category
print("Mortality Risk Distribution:")
print(df['Mortality_Risk'].value_counts())

Mortality Risk Distribution:
Mortality_Risk
Low       1584
Medium    1584
High      1584
Name: count, dtype: int64


### Step 3: Temporal Feature Engineering (Lags & Growth)

In [5]:
# TEMPORAL FEATURE ENGINEERING
# Capturing the delayed impact of economic shocks on health outcomes

# 1. SORT DATA: Crucial for correct grouping and shifting
df = df.sort_values(by=['Country Name', 'Year'])

# 2. CREATE LAG FEATURES (Shift by 1 Year)
# We shift Food Prices and GDP to see their impact on the FOLLOWING year's mortality
df['Food_Price_Lag1'] = df.groupby('Country Name')['Food_Price_Index'].shift(1)
df['GDP_Lag1'] = df.groupby('Country Name')['GDP_per_capita'].shift(1)

# 3. CREATE GROWTH RATES
# Measuring the speed of economic change (Percentage Change)
df['GDP_Growth_Rate'] = df.groupby('Country Name')['GDP_per_capita'].pct_change()

# 4. EFFICIENCY PROXY
# Ratio of health spending to mortality (Lower is better/more efficient)
df['Health_Efficiency_Index'] = df['Health_Expenditure'] / (df['Mortality_Rate'] + 1)

# 5. HANDLING MISSING VALUES
# Shifting data creates NaNs in the first year of each country's data
print(f"Missing values after engineering:\n{df[['Food_Price_Lag1', 'GDP_Growth_Rate']].isnull().sum()}")

# Dropping NaNs to keep the dataset clean for the ML model
df = df.dropna().reset_index(drop=True)

# PREVIEW ENGINEERED FEATURES
df[['Country Name', 'Year', 'Mortality_Rate', 'Food_Price_Lag1', 'GDP_Growth_Rate']].head()

Missing values after engineering:
Food_Price_Lag1    202
GDP_Growth_Rate    202
dtype: int64


,Country Name,Year,Mortality_Rate,Food_Price_Lag1,GDP_Growth_Rate
0,Afghanistan,2003,10.714,16.00000,0.111297
1,Afghanistan,2004,10.375,11.00000,0.115112
2,Afghanistan,2005,9.913,12.40000,0.146194
3,Afghanistan,2006,9.659,12.66875,0.078818
4,Afghanistan,2007,9.487,13.33750,0.371983


### Step 4: Categorical Encoding

In [6]:
# LABEL ENCODING
# Converting the Target classes into numerical format for the Classifier
le = LabelEncoder()
df['Risk_Label'] = le.fit_transform(df['Mortality_Risk'])

# Create a mapping for future reference (Inference/Dashboard)
risk_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Risk Level Mapping:", risk_mapping)

# Check the final shape after cleaning NaNs
print(f"Final Dataset Shape for Modeling: {df.shape}")

Risk Level Mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Final Dataset Shape for Modeling: (4550, 14)


### Step 5: Feature Selection & Data Splitting

In [7]:
# FEATURE SELECTION
# Choosing predictors that capture economic, health, and temporal signals
features = [
    'GDP_per_capita', 
    'Health_Expenditure', 
    'Food_Price_Index', 
    'Homicide_Rate',
    'Food_Price_Lag1', 
    'GDP_Lag1', 
    'GDP_Growth_Rate', 
    'Health_Efficiency_Index'
]

X = df[features]
y = df['Risk_Label']

# DATA SPLITTING
# 80% for training the model, 20% for testing its "unseen" performance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (3640, 8)
Testing set size: (910, 8)


### Step 6: Saving the Prepared Data (Crucial)

In [8]:
# SAVING PREPARED DATA
# Exporting as NumPy arrays or CSV for the next notebook
import joblib

# Saving the split data and the label encoder for consistency in Notebook 4
data_to_save = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'features': features,
    'le': le
}

joblib.dump(data_to_save, 'data_for_modeling.pkl')
print("Success: Prepared data and LabelEncoder saved as 'data_for_modeling.pkl'")

Success: Prepared data and LabelEncoder saved as 'data_for_modeling.pkl'


###  Notebook 3 Wrap-up
- Created a 3-tier **Mortality Risk** target variable.
- Engineered **Temporal Lags** to capture delayed economic impacts.
- Developed **Efficiency Indices** for health expenditure.
- Prepared and saved a structured dataset with **4,550 samples** for machine learning.

**Next Step:** Proceed to **Notebook 4: Random Forest Modeling & Evaluation**.